In [ ]:
"This IPYNB file is only used as a test, and a tool for deploying the model. The real Code goes in the same file, but "
"with a .py extension rather than a .ipynb"

In [ ]:
from pathlib import Path
import pandas as pd

# Try common working directories (notebook folder and workspace root).
candidate_paths = [
    Path("../../../../SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
    Path("backend/SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate usgs_data_USGS-01646500.csv")
print(f"Using CSV file at: {csv_path}")

df = pd.read_csv(csv_path)
df

In [ ]:
# Flood Action Stage: 5 ft
# Minor Flood Stage: 10 ft
# Moderate Flood Stage: 12 ft
# Major Flood Stage: 14 ft
FLOOD_ACTION_STAGE = 5.0
MINOR_FLOOD_STAGE = 10.0
MODERATE_FLOOD_STAGE = 12.0
MAJOR_FLOOD_STAGE = 14.0

#df.hist(figsize=(10, 6))
# get the instances where Gage Height is > 5
df = df.dropna(subset=['gage_height_ft'])
df_flood = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE]
df_minor_flood = df[df['gage_height_ft'] > MINOR_FLOOD_STAGE]
df_moderate_flood = df[df['gage_height_ft'] > MODERATE_FLOOD_STAGE]
df_major_flood = df[df['gage_height_ft'] > MAJOR_FLOOD_STAGE]

# print the lengths of each of the dataframes
print(f"Total records: {len(df)}")
print(f"Flood Action Stage records: {len(df_flood)}")
print(f"Minor flood records: {len(df_minor_flood)}")
print(f"Moderate flood records: {len(df_moderate_flood)}")
print(f"Major flood records: {len(df_major_flood)}")

In [ ]:

# Name the 'Unnamed: 0' column as 'datetime' and convert it to datetime type
df['datetime'] = pd.to_datetime(df['Unnamed: 0'])

df = df.sort_values('datetime').reset_index(drop=True)

# how many hours of gap counts as "the storm ended" (tune this to your data's
# sampling frequency, e.g. 6-12h for hourly gauge data, 24-48h for daily)
GAP_HOURS = 12

# isolate just the flagged (action-stage) rows
flood_rows = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE].copy()

# time since previous flagged reading, saved in column 'gap'
flood_rows['gap'] = flood_rows['datetime'].diff()

# start a new event whenever the gap exceeds threshold (or it's the first row)
flood_rows['new_event'] = (
    flood_rows['gap'].isna() | (flood_rows['gap'] > pd.Timedelta(hours=GAP_HOURS))
)
flood_rows['event_id'] = flood_rows['new_event'].cumsum()

df = df.merge(
    flood_rows[['datetime', 'event_id']],
    on='datetime',
    how='left'
)

# summarize each event
events = flood_rows.groupby('event_id').agg(
    start=('datetime', 'min'),
    end=('datetime', 'max'),
    n_readings=('datetime', 'count'),
    peak_gage_height=('gage_height_ft', 'max')  # adjust column name as needed
).reset_index(drop=True)

events['duration_hours'] = (events['end'] - events['start']).dt.total_seconds() / 3600

print(f"Total flagged readings: {len(flood_rows)}")
print(f"Independent storm events: {len(events)}")
print(events)


In [ ]:
LOOKAHEAD_HOURS = 24  # predict within next 24h
FREQ_MINUTES = 15     # Hydraulic data's sampling interval 

# Cqalculate how many rows ahead
lookahead_steps = int(LOOKAHEAD_HOURS * 60 / FREQ_MINUTES)

# 'will_flood' checks whether the flood stage will be exceeded in the next `lookahead_steps` readings, setting it as 
# 1 if any of the next readings exceed the flood action stage, and 0 otherwise. This is used as the target variable for the model.

df['will_flood'] = (
    df['gage_height_ft']
    .shift(-1)                                   
    .rolling(window=lookahead_steps, min_periods=1)
    .max()
    .shift(-(lookahead_steps - 1))                # align window to start right after current row
    > FLOOD_ACTION_STAGE
).astype(int)



In [ ]:
import numpy as np

# make a column called gage_height_roc_1h and gage_height_roc_6h to see the rate of change in gage height over 1 hour and 6 hours, respectively 
df['gage_height_roc_1h'] = df['gage_height_ft'].diff(int(60/FREQ_MINUTES))
df['gage_height_roc_6h'] = df['gage_height_ft'].diff(int(360/FREQ_MINUTES))

df['gage_height_now'] = df['gage_height_ft']
df['streamflow_now'] = df['streamflow_cfs']

for col in ['precip_3hr', 'precip_24hr', 'precip_72hr', 'turbidity_fnu']:
    if col in df.columns:
        df[f'{col}_log'] = np.log1p(df[col].clip(lower=0))

feature_columns = [
    # hydraulic — current state + trend
    'gage_height_ft',
    'gage_height_roc_1h',
    'gage_height_roc_6h',

    # precip — log-transformed versions only (not the raw skewed ones)
    'precip_3hr_log',
    'precip_24hr_log',
    'precip_72hr_log',

    # weather
    'temperature_2m',
    'wind_speed_10m',
    'vapour_pressure_deficit',
    'rain',
    'snowfall',
    'snow_depth',

    # water quality
    'specific_conductance_us_cm',
    'temperature_c',
]


In [ ]:
# tag every row (not just flood rows) with which event's "influence window" it falls in
# so the same storm doesn't appear in both train and test
events_sorted = events.sort_values('start').reset_index(drop=True)

# hold out the most recent ~20% of events as test
n_test_events = int(len(events_sorted) * 0.2)
test_events = events_sorted.iloc[-n_test_events:]
train_events = events_sorted.iloc[:-n_test_events]

test_start_cutoff = test_events['start'].min() - pd.Timedelta(days=3)  # buffer before first test storm

train_df = df[df['datetime'] < test_start_cutoff].dropna(subset=feature_columns + ['will_flood'])
test_df  = df[df['datetime'] >= test_start_cutoff].dropna(subset=feature_columns + ['will_flood'])

test_df

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


X_train, y_train = train_df[feature_columns], train_df['will_flood']
X_test, y_test = test_df[feature_columns], test_df['will_flood']

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
probs = model.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    classification_report, roc_auc_score
)

probs = model.predict_proba(X_test)[:, 1]

# PR-AUC (Precision-Recall Area Under Curve) is used for imbalanced datasets, as it focuses on the performance
#  of the positive class (flood events) and is more informative than accuracy in such cases.
print(f"PR-AUC: {average_precision_score(y_test, probs):.3f}")

# ROC-AUC (Receiver Operating Characteristic Area Under Curve) is a performance measurement for classification 
# prediction problems at various threshold settings. It tells how much the model is capable of distinguishing
print(f"ROC-AUC: {roc_auc_score(y_test, probs):.3f}")

# don't default to 0.5 — pick a threshold that favors recall (missing a flood is worse)
precision, recall, thresholds = precision_recall_curve(y_test, probs)


In [ ]:
def lead_time_last_rise(event_rows, flood_start, threshold=0.5, min_below_hours=6, freq_minutes=15):
    """
    Find the most recent rise above threshold before the flood, requiring
    the probability to have dipped below threshold for at least
    `min_below_hours` beforehand — this separates a fresh, distinct rise
    from an unrelated earlier event or a brief blip.
    """
    trace = event_rows.sort_values('datetime').reset_index(drop=True)
    above = trace['predicted_prob'] >= threshold
    min_below_rows = int(min_below_hours * 60 / freq_minutes)

    crossings = trace.index[above & ~above.shift(1, fill_value=False)]

    valid_crossings = []
    for idx in crossings:
        if idx < min_below_rows:
            continue  # not enough prior history in this window to confirm a real dip — skip, don't assume
        lookback_start = idx - min_below_rows
        if not above.iloc[lookback_start:idx].any():
            valid_crossings.append(idx)

    if not valid_crossings:
        return None  # genuinely no valid crossing found — e.g. sustained risk the whole window

    last_idx = valid_crossings[-1]
    return trace.loc[last_idx, 'datetime']

In [ ]:

test_df = test_df.copy()

# Add the predicted probabilities to the test dataframe for further analysis
test_df['predicted_prob'] = probs

ALERT_THRESHOLD = 0.716

# This finds all unique event IDs where the gage height exceeds the flood action stage, indicating a flood event.
flood_events = test_df.loc[test_df['gage_height_ft'] > FLOOD_ACTION_STAGE, 'event_id'].dropna().unique()

results = []

LOOKBACK_HOURS = 168  # how far before the flood to look for an early alert

results = []
for eid in flood_events:
    flood_rows_this_event = test_df[test_df['event_id'] == eid]
    flood_start = flood_rows_this_event['datetime'].min()
    window_start = flood_start - pd.Timedelta(hours=LOOKBACK_HOURS)
    event_rows = test_df[(test_df['datetime'] >= window_start) & (test_df['datetime'] <= flood_rows_this_event['datetime'].max())]
    alert_time = lead_time_last_rise(event_rows, flood_start)
    lead_time = (flood_start - alert_time).total_seconds() / 3600 if alert_time is not None else None

    results.append({'event_id': eid, 'lead_time_hours': lead_time, 'peak_prob': flood_rows_this_event['predicted_prob'].max()})


results_df = pd.DataFrame(results)
print(results_df)
print(f"\nMedian lead time: {results_df['lead_time_hours'].median():.1f}h")

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, probs)

# find threshold that gives recall >= some target, e.g. 0.90
import numpy as np
target_recall = 0.90
qualifying = np.where(recall >= target_recall)[0]
idx = qualifying[-1] if len(qualifying) > 0 else -1  # take the LAST (highest-threshold) index, not the first

print(f"Threshold for {target_recall} recall: {thresholds[idx]:.3f}, precision at that point: {precision[idx]:.3f}")

In [ ]:
from sklearn.metrics import confusion_matrix
final_threshold = thresholds[idx]
preds = (probs >= final_threshold).astype(int)
print(confusion_matrix(y_test, preds))